# PLOT-DAS walkthrough — IOI (attention-head cells)

End-to-end run of the PLOT-DAS pipeline on **cell 14 — IOI × GPT-2 × output_position**, plus the **E-I-2 diagnostic** that found PLOT picks the wrong heads.

Cell 14 is structurally different from the other MIB tasks:

- Ships **attention-head featurizers**, not residual-stream
- Uses **`PatchAttentionHeads` joint-mode DAS** with MSE-on-logit-diff loss
- Has a separate eval entry point (`evaluate_ioi_submission_task`)
- Requires a **linear-params bootstrap** step (compute `bias`, `token_coeff`, `position_coeff` for the DAS target) before any PLOT run

Cell 14's PLOT-shipped MSE is 16.0 — way worse than DAS leaderboard 2.20. The diagnostic at the end of this notebook (E-I-2) shows the right heads exist: bypassing PLOT and using literature-known S-Inhibition heads cuts MSE to 4.12. This is PLOT_SHORTCOMINGS §13 in action.

Reproducing via CLI:

```bash
# One-time bootstrap (already saved if you've run before)
.venv-mib/bin/python -c "
from mib_submission.ioi import bootstrap_linear_params
bootstrap_linear_params('gpt2')
"

# Run the cell
.venv-mib/bin/python -m mib_submission.plot.run \\
    --task ioi_task --model gpt2 --variable output_position --dataset-size 512

.venv-mib/bin/python scripts/eval_cell.py \\
    --cell ioi_task_GPT2LMHeadModel_output_position
```

Runtime: ~30 min on 8 GB GPU (GPT-2 small is light).

In [ ]:
import sys, subprocess, json
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / 'mib_submission').is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO))

import torch

from mib_submission.pipeline import setup_attention_head_experiment, MIB_TRACK
from mib_submission.plot.configs import default_config
from mib_submission.plot.pipeline import select_sites_via_plot
from mib_submission.ioi import (
    bootstrap_linear_params, load_linear_params, ensure_linear_params_json,
    write_ioi_submission, LINEAR_PARAMS_FILENAME,
)
from mib_submission.ioi.bootstrap import model_short_name
from mib_submission.ioi._patches import (
    patch_lm_pipeline_load, patch_model_config_head_dim,
)

DTYPE = torch.float16
SUBMISSION_ROOT = REPO / 'submissions' / 'plot'
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
patch_lm_pipeline_load()  # idempotent

## 1. Bootstrap the linear params

The IOI DAS loss isn't cross-entropy on the answer token — it's **MSE on logit-difference** (the difference in logit scores between the correct vs incorrect name). The target logit-diff for an interchanged example is a learned linear function:

$$\text{target\_diff} = \text{bias} + \text{token\_coeff} \cdot \text{token\_flip} + \text{position\_coeff} \cdot \text{position\_flip}$$

where `token_flip ∈ {-1, +1}` and `position_flip ∈ {-1, +1}` indicate whether the source counterfactual flips the IO token and/or IO position.

The three coefficients are learned once per model via linear regression over 4 IOI splits (the bootstrap step). They're saved to `submissions/plot/ioi_linear_params.json` and read at DAS-train time.

This already ran once for GPT-2 — you'll see it skip if `ioi_linear_params.json` exists.

In [ ]:
params_path = SUBMISSION_ROOT / LINEAR_PARAMS_FILENAME

if not params_path.is_file():
    print('Bootstrapping IOI linear params for GPT-2...')
    bootstrap_linear_params('gpt2', output_file=str(params_path))
else:
    print(f'Linear params already at {params_path}')

linear_params = load_linear_params(params_path, model_short='gpt2')
print(f'bias = {linear_params.bias:.4f}')
print(f'token_coeff = {linear_params.token_coeff:.4f}')
print(f'position_coeff = {linear_params.position_coeff:.4f}')
print(f'R² = {linear_params.score:.4f}')

## 2. Build the attention-head experiment bundle

Different from residual-stream cells — we call `setup_attention_head_experiment` instead of `setup_residual_experiment`. The bundle's `model_units_lists` contains one `AttentionHead` per `(layer, head)` pair across all candidate layers.

GPT-2-small: 12 layers × 12 heads = 144 candidate sites.

Token positions in IOI: a single `'all'` (intervene on the full head output, not a specific sequence position — see `mib_submission/ioi/` for IOI-specific token-position handling).

In [ ]:
config = default_config(
    task='ioi_task',
    model_name='gpt2',
    variable='output_position',
    overrides={'dataset_size': 512},
)

print(f'cell: {config.task} × {config.model_class_name} × {config.variable}')
print(f'plot_config.variables (OT rows = counterfactual splits): {config.plot_config.variables}')
print(f'plot_config.per_row_split_datasets: {config.plot_config.per_row_split_datasets}')
print(f'DAS: n_features={config.n_features}, epochs={config.training_epochs}')

from transformers import AutoConfig
model_cfg = AutoConfig.from_pretrained(config.model_name)
n_layers = model_cfg.num_hidden_layers
n_heads = getattr(model_cfg, 'num_attention_heads', None) or model_cfg.n_head
layer_head_list = [(L, H) for L in range(n_layers) for H in range(n_heads)]
print(f'\ncandidate sites: {n_layers}L × {n_heads}H = {len(layer_head_list)}')

bundle = setup_attention_head_experiment(
    model_name=config.model_name,
    layer_head_list=layer_head_list,
    target_variables=[config.variable],
    linear_params=linear_params.as_dict(),
    dtype=DTYPE,
    dataset_size=config.dataset_size,
    config_overrides={
        'training_epoch': config.training_epochs,
        'init_lr': config.init_lr,
        'n_features': config.n_features,
        'batch_size': config.train_batch_size,
    },
    verbose=True,
    per_site_units=True,
)
patch_model_config_head_dim(bundle.pipeline.model.config)
print(f'\ntrain splits: {sorted(bundle.train_data.keys())}')

## 3. Run PLOT site selection

Same `select_sites_via_plot` call as for residual-stream cells, but the per-row signature collection runs against **per-row split datasets** rather than a single shared split:

- Row 0 (`s1_io_flip`): bases with source counterfactual that swaps S1 with IO
- Row 1 (`s2_io_flip`): bases with source that swaps S2 with IO
- Row 2 (`s1_ioi_flip_s2_ioi_flip`): bases that swap both

Stage A picks layers from the per-row OT plans (with sites = (layer, head) pairs collapsed per layer for the layer-level aggregation). Stage B then picks specific heads within the picked layers.

In [ ]:
fit_split = sorted(bundle.train_data.keys())[0]
print(f'running PLOT on per-row split scheme {config.plot_config.per_row_split_datasets!r}\n')

selection = select_sites_via_plot(
    bundle,
    bundle.train_data[fit_split],
    config=config.plot_config,
    verbose=True,
)

a_eps, a_topk, a_score = selection.stage_a_chosen
b_eps, b_topk, b_score = selection.stage_b_chosen
print(f'\nStage A: eps={a_eps} top_k={a_topk} IIA={a_score:.4f}')
print(f'Stage A picked layers: {selection.stage_a_layers}')
print(f'Stage B: eps={b_eps} top_k={b_topk} IIA={b_score:.4f}')
print(f'\nPicked attention heads:')
picked_heads = []
for s in selection.selected_sites:
    if len(s) == 3:
        L, H, tok = s
        picked_heads.append((int(L), int(H)))
        print(f'  L{L}H{H}/{tok}')

## 4. Joint DAS training at picked heads

PLOT's contribution ends; we now hand off to MIB's DAS. For IOI specifically, the training uses **joint mode** — one shared rotation across all picked heads' concatenated subspace, not separate rotations per head. This matches the IOI baseline's setup.

The DAS objective: minimize MSE between (a) the model's interchange-intervention output's logit-diff and (b) the linear-target logit-diff (using the bootstrapped `bias`, `token_coeff`, `position_coeff`).

PLOT's typical picks for cell 14: L9 Name Movers (e.g. L9H1, L9H5, L9H10). DAS training metrics will show MSE on the train splits — expect noisy values around 16-30 because L9 carries token info, not position info.

In [ ]:
from mib_submission.pipeline import add_mib_to_syspath
add_mib_to_syspath()
from experiments.attention_head_experiment import PatchAttentionHeads
sys.path.insert(0, str(MIB_TRACK / 'baselines' / 'ioi_baselines'))
from ioi_utils import ioi_loss_and_metric_fn, checker as ioi_checker

das_config = {
    'evaluation_batch_size': config.eval_batch_size,
    'batch_size': config.train_batch_size,
    'training_epoch': config.training_epochs,
    'check_raw': True,
    'n_features': config.n_features,
    'regularization_coefficient': 0.0,
    'output_scores': True,
    'shuffle': True,
    'temperature_schedule': (1.0, 0.01),
    'init_lr': config.init_lr,
    'loss_and_metric_fn':
        lambda pipe, intervenable, batch, units:
            ioi_loss_and_metric_fn(pipe, intervenable, batch, units),
}
das_experiment = PatchAttentionHeads(
    pipeline=bundle.pipeline,
    causal_model=bundle.causal_model,
    layer_head_list=picked_heads,
    token_positions=bundle.token_positions,
    checker=lambda logits, params: ioi_checker(logits, params, bundle.pipeline),
    config=das_config,
)
das_experiment.train_interventions(
    bundle.train_data, [config.variable], method='DAS', verbose=True,
)

## 5. Write IOI submission + eval

IOI submission layout is **flat** (not nested as in the IOI example notebook in MIB's repo). `write_ioi_submission` handles the layout + sibling `ioi_linear_params.json`.

In [ ]:
short = model_short_name(config.model_class_name)
cell_dir = write_ioi_submission(
    SUBMISSION_ROOT,
    experiment=das_experiment,
    model_class_name=config.model_class_name,
    variable=config.variable,
    overwrite=True,
)
ensure_linear_params_json(
    SUBMISSION_ROOT, model_short=short,
    model_class_name=config.model_class_name, params=linear_params,
)
print(f'wrote to {cell_dir}')
for f in sorted(cell_dir.iterdir()):
    print(f'  {f.name}')

In [ ]:
cell = cell_dir.name
result = subprocess.run(
    [sys.executable, str(REPO / 'scripts' / 'eval_cell.py'), '--cell', cell],
    capture_output=True, text=True,
)
for line in result.stdout.strip().split('\n')[-12:]:
    print(line)

## 6. Per-site MSE breakdown

IOI eval reports MSE on logit-difference (lower is better). DAS baseline for cell 14 = 2.20. PLOT typically lands at ~16.0 — driven by 22+ MSE on the position-flip splits (`s1_io_flip_test`, `s2_io_flip_test`). The `s1_ioi_flip_s2_ioi_flip_test` split happens to be lower (~2.8) because BOTH position and token flip simultaneously.

In [ ]:
results_files = list(cell_dir.glob('*results.json'))
d = json.loads(results_files[0].read_text())
splits = list(d['dataset'].keys())
sites = list(d['dataset'][splits[0]]['model_unit'].keys())

# IOI eval reports one joint key for all picked heads
for site in sites:
    short_id = site.split("id='")[1].split("'")[0]
    print(f'site: {short_id}\n')
    row = []
    for s in splits:
        sv = d['dataset'][s]['model_unit'][site]
        for k, v in sv.items():
            if isinstance(v, dict) and 'average_score' in v:
                row.append((s, v['average_score'])); break
    for split_name, mse in row:
        print(f'  {split_name:35s}: MSE {mse:.4f}')
    mean_mse = sum(m for _, m in row) / len(row)
    print(f'  {"mean MSE":35s}: {mean_mse:.4f}')
    print()

print('reference: DAS leaderboard MSE 2.20; our shipped MSE ~16.0')

## 7. Diagnostic — E-I-2: bypass to S-Inhibition heads

PLOT's 16.0 MSE is structural — the signature design (logit-diff effect) systematically picks heads with *direct* effects on output (L9 Name Movers), not heads that route information *indirectly* (L7-L8 S-Inhibition heads). The latter are exactly where IOI position info lives.

Diagnostic: hardcode the literature-known S-Inhibition heads as PLOT's picks (bypass Stage A/B), train DAS there, see what happens.

From the diagnostic session (PLOT_SHORTCOMINGS §13):

| split | PLOT pick (L9 Name Movers) | S-Inhibition bypass (L7H3, L7H9, L8H6, L8H10) |
|---|---|---|
| s1_io_flip_test | 22.95 | **6.08** |
| s2_io_flip_test | 22.28 | **3.93** |
| s1_ioi_flip_s2_ioi_flip_test | 2.79 | 2.36 |
| **mean** | **16.0** | **4.12** |

Closing the gap from 16.0 to 4.12 just by changing which heads PLOT trains DAS at. The DAS baseline at 2.20 is still better — there's a residual ~2 MSE gap from training-data quantity — but the dominant gap is from PLOT picking the wrong heads.

Reproduce with the `--bypass-sites` flag (supports the IOI 3-tuple format):

```bash
.venv-mib/bin/python -m mib_submission.plot.run \\
    --task ioi_task --model gpt2 --variable output_position \\
    --dataset-size 512 \\
    --bypass-sites '7:3,7:9,8:6,8:10'
```

We didn't ship the bypass version as cell 14's submission — it's a hand-picked-heads result, not pure PLOT. The bypass-result submission is preserved at `submissions/_plot_backups/ioi14_pre_e_i_2_PLOT_picks_*` for reference.

## Open question

Closing the IOI gap properly would require a different *signature* (not a different solver). The current per-site signature aggregates direct effects on output logit-diff. A signature that captures cascading/indirect effects — e.g. ablation cascades that break downstream Name Movers — would surface S-Inhibition heads naturally. This is `H-IOI-NEW-1` in `HYPOTHESES.md`.